# Momentum-sector spin model

This example describes an independent one-magnon band of a periodic spin chain. It demonstrates:

- a physical dispersion relation evaluated on a Brillouin-zone grid;
- one independent solver sector for every momentum $k$;
- statistical $k$ weights carried by the initial state;
- momentum-resolved and momentum-averaged linear spectroscopy;
- a third-order rephasing spectrum calculated from the same sectorized model.

Natural units are used, with $\hbar=1$. Energies are in eV and times are in eV$^{-1}$.

In [ ]:
from pathlib import Path
import sys

import numpy as np

# Locate the project root when the notebook is launched from examples/
# or from another directory inside the repository.
current_directory = Path.cwd().resolve()
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "projet_solver10" / "__init__.py").is_file():
        project_root = str(candidate)
        if project_root not in sys.path:
            sys.path.insert(0, project_root)
        break
else:
    raise RuntimeError("Could not locate the directory containing projet_solver10.")

from projet_solver10 import (
    EigenbasisKModel,
    FrequencyPathway,
    PropagationInterval,
    SpectroscopyPlotter,
    SpectroscopyProtocol,
    SpectroscopySolver,
    standard_nq_protocol,
)

## 1. One-magnon dispersion

In the one-excitation sector of a nearest-neighbor spin-flip model, translation symmetry produces momentum eigenstates. We use the simple cosine band

$$\omega(k)=\omega_0+2J\cos k,\qquad -\pi\leq k<\pi.$$

Each momentum block contains a reference state $|0_k\rangle$ and one spin-flip state $|1_k\rangle$. This is an independent-quasiparticle model: the current adapter does not include scattering between different momenta or magnon-magnon interactions.

In [ ]:
def magnon_dispersion(k, omega_0, exchange):
    return omega_0 + 2.0 * exchange * np.cos(k)


n_k = 17
omega_0 = 1.55
exchange = -0.08
dipole_strength = 1.0

k_points = np.linspace(-np.pi, np.pi, n_k, endpoint=False)
magnon_energies = magnon_dispersion(k_points, omega_0, exchange)
k_weights = np.ones(n_k) / n_k

print(f"Momentum sectors: {n_k}")
print(f"Magnon band: {magnon_energies.min():.4f} to {magnon_energies.max():.4f} eV")
print(f"Sum of k weights: {k_weights.sum():.6f}")

## 2. Build the momentum blocks

For each $k$,

$$
H(k)=\begin{pmatrix}0&0\\0&\omega(k)\end{pmatrix},
\qquad
J(k)=\mu\begin{pmatrix}0&1\\1&0\end{pmatrix}.
$$

A local transverse spin probe has a broad momentum content, so all momentum sectors are assigned the same transition amplitude here. A spatially uniform probe would instead impose its own momentum-selection rule, often selecting only $k=0$.

The named observables `k_00`, `k_01`, and so on act as momentum filters. Each one contains the dipole operator in one sector and zeros in every other sector. They let us recover all momentum-resolved contributions after a single propagation at each frequency.

In [ ]:
H_stack = np.zeros((n_k, 2, 2), dtype=complex)
H_stack[:, 1, 1] = magnon_energies

sigma_x = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
interaction_stack = np.repeat(
    (dipole_strength * sigma_x)[np.newaxis, :, :],
    n_k,
    axis=0,
)

k_observable_names = tuple(f"k_{index:02d}" for index in range(n_k))
observable_arrays = {"polarization": interaction_stack}
for index, name in enumerate(k_observable_names):
    momentum_filter = np.zeros_like(interaction_stack)
    momentum_filter[index] = interaction_stack[index]
    observable_arrays[name] = momentum_filter

print("Hamiltonian stack shape:", H_stack.shape)
print("Number of momentum-filter observables:", len(k_observable_names))

## 3. Sectorized model and sparse backend

`EigenbasisKModel` maps each slice of `H_stack` to a separate sector named `k0`, `k1`, and so on. It returns only intra-sector Hamiltonian and transition blocks, so propagation never transfers amplitude between different momenta.

The normalized `k_weights` are stored in the traces of the initial density blocks. Consequently, tracing the final polarization automatically returns the Brillouin-zone average. The sparse-sector backend retains this block organization throughout the calculation.

In [ ]:
model = EigenbasisKModel(
    H_stack,
    interaction_stack,
    detection_op_array=interaction_stack,
    observable_op_arrays=observable_arrays,
    k_weights=k_weights,
)

solver = SpectroscopySolver(backend="sparse_sector", eta=0.012)
solver.feed_model(model)
summary = solver.summary()

assert len(model.sectors()) == n_k
assert model.transition_decomposition() == "automatic_energy"
assert summary["total_dimension"] == 2 * n_k

print("Backend:", summary["backend"])
print("First five sector labels:", summary["sectors"][:5])
print("Total direct-sum dimension:", summary["total_dimension"])
print("Transition decomposition:", model.transition_decomposition())

## 4. Momentum-resolved linear response

A single ket-side raising interaction creates the optical coherence. For every trial frequency, the solver propagates the complete sectorized state once and then evaluates the total polarization together with all momentum-filter observables.

The total signal must therefore equal the sum of the momentum-resolved signals, including their statistical weights.

In [ ]:
linear_pathway = FrequencyPathway(
    name="linear",
    interactions=("Ku",),
    component="linear",
)
linear_protocol = SpectroscopyProtocol(
    intervals=(PropagationInterval("omega", "frequency", coherence_order=1),),
    name="linear_frequency",
)
solver.set_pathways((linear_pathway,))

omega = np.linspace(1.32, 1.78, 121)
linear_total = np.zeros(omega.size, dtype=complex)
linear_by_k = np.zeros((n_k, omega.size), dtype=complex)

for frequency_index, frequency in enumerate(omega):
    point = solver.calc_pathway_observables(
        linear_pathway,
        linear_protocol,
        {"omega": frequency},
        observables=k_observable_names,
    )
    linear_total[frequency_index] = point.value
    for momentum_index, name in enumerate(k_observable_names):
        linear_by_k[momentum_index, frequency_index] = point.observables[name]

reconstruction_residual = np.max(np.abs(linear_total - linear_by_k.sum(axis=0)))
peak_energies = omega[np.argmax(np.abs(linear_by_k), axis=1)]
maximum_peak_error = np.max(np.abs(peak_energies - magnon_energies))
frequency_step = omega[1] - omega[0]

assert reconstruction_residual < 1e-10
assert maximum_peak_error <= frequency_step
print(f"Momentum-sum residual: {reconstruction_residual:.3e}")
print(f"Largest dispersion-peak error: {maximum_peak_error:.4e} eV")

In [ ]:
plotter = SpectroscopyPlotter(detection_phase=0.0)

linear_plot = plotter.plot_1d(
    linear_total,
    w=omega,
    params={
        "view": "all",
        "title": "Brillouin-zone-averaged linear response",
        "xlabel": r"Energy $\omega$ (eV)",
        "ylabel": "Signal",
    },
)

momentum_map = plotter.plot_contourf_multi_spectra(
    [linear_by_k],
    x_values=omega,
    y_values=k_points,
    title_list=["Momentum-resolved linear response"],
    params={
        "view": "abs",
        "normalization": "none",
        "labels": (r"Energy $\omega$ (eV)", r"Momentum $k$"),
        "aspect": "auto",
        "diagonals": False,
        "style": {"abs_cmap": "magma", "levels": 35, "contour_lines": False},
    },
)

## 5. Third-order rephasing response

The same momentum sectors can be used in nonlinear spectroscopy. For an independent two-level transition at every $k$, the two rephasing pathways are

$$R_1=(B_u,K_u,B_d),\qquad R_2=(B_u,B_d,K_u),$$

with coherence history $q=(-1,0,+1)$. There is no excited-state-absorption pathway because each momentum block contains only $|0_k\rangle$ and $|1_k\rangle$.

In [ ]:
pathway_r1 = FrequencyPathway(
    name="R1",
    interactions=("Bu", "Ku", "Bd"),
    component="rephasing",
)
pathway_r2 = FrequencyPathway(
    name="R2",
    interactions=("Bu", "Bd", "Ku"),
    component="rephasing",
)

protocol_1q = standard_nq_protocol(
    order=1,
    nq_interval=1,
    detection_interval=3,
    n_interactions=3,
    nq_axis="omega_1q",
    detection_axis="omega_emit",
)

omega_1q = np.linspace(-1.78, -1.32, 17)
omega_emit = np.linspace(1.32, 1.78, 17)
result_rephasing = solver.generate_spectrum(
    protocol_1q,
    axes={"omega_1q": omega_1q, "omega_emit": omega_emit},
    fixed_coordinates={"t2": 0.0},
    pathways=(pathway_r1, pathway_r2),
)

rephasing = result_rephasing.components["rephasing"]
assert np.all(np.isfinite(rephasing))
assert np.max(np.abs(rephasing)) > 0.0

peak_index = np.unravel_index(np.argmax(np.abs(rephasing)), rephasing.shape)
print("Strongest rephasing coordinate:")
print(f"  omega_1q  = {omega_1q[peak_index[0]]:.4f} eV")
print(f"  omega_emit = {omega_emit[peak_index[1]]:.4f} eV")

In [ ]:
rephasing_plot = plotter.plot_spectrum_result(
    result_rephasing,
    params={
        "source": "components",
        "names": ["rephasing"],
        "view": "all",
        "normalization": "row",
        "labels": (r"Emission energy $\omega_{\mathrm{emit}}$ (eV)", r"Excitation energy $\omega_{1Q}$ (eV)"),
        "title": r"Momentum-averaged $\chi^{(3)}$ rephasing spectrum",
        "diagonals": "auto",
        "style": {
            "cmap": "RdYlBu_r",
            "abs_cmap": "magma",
            "levels": 30,
            "contour_lines": False,
        },
    },
)

## 6. What this example establishes

1. A momentum label can be represented directly as a solver sector.
2. The Hamiltonian, transition operator, state weight, and observables may all depend on $k$.
3. Momentum-filter observables reconstruct the weighted total response without repeating propagation.
4. The sparse backend preserves independent blocks instead of assembling one dense Liouville matrix.
5. The same sectorized model supports both linear and nonlinear spectroscopy.

This construction represents independent momentum blocks. Processes that transfer momentum, such as disorder scattering, phonon scattering, or magnon-magnon interactions, require explicit inter-sector coupling in a more general `SectorModel`.